In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px

import sys
sys.path.append('../')
import plotting
import model

# Read abundance data from GCall mapping

In [ ]:
pool_df = model.read_csvs("../data/internal_datasets/GCall/", keep_all_sequences=True)
pool_df['seq_id'] = pool_df.index.to_series().apply(lambda x: "#" + x[1:])
pool_df

# melt the dataframe
pool_df = pool_df.melt(id_vars="seq_id")
pool_df['n_cycles'] = pool_df.variable.apply(lambda x: float(15*int(x[-1])))
pool_df

# Get the parameters estimated by the model fit

In [ ]:
params = pd.read_csv("../data/internal_datasets/GCall/params.csv", dtype={"seq_id": str})
params['seq_id'] = params.seq_id.apply(lambda x: "#" + x[1:])
params

# create figures per efficiency group

In [ ]:
selected_x0 = [params['x0'].quantile(i) for i in (0.09, 0.3, 0.5, 0.7, 0.91)]

colormap = {
    '0': '#3182bd',
    '1': '#31a354',
    '2': '#e6550d',
    '3': '#756bb1',
    '4': '#636363',
}
colormap_light = {
    '0': '#bdd7e7',
    '1': '#bae4b3',
    '2': '#fdbe85',
    '3': '#cbc9e2',
    '4': '#cccccc',
}


for efficiency_name, efficiency_thresholds in [('low', (0.97, 0.99)), ('medium', (0.9975, 1.0025)), ('high', (1.005, 1.02))]:

    # select sequences with a certain efficiency
    iparams = params.loc[(params['eff'] > efficiency_thresholds[0]) & (params['eff'] < efficiency_thresholds[1])].copy()
    iparams['group'] = efficiency_name

    # create a new dataframe with the rows that have the closest x0 values to the selected x0 values
    selected_params = pd.DataFrame()
    for i, x0 in enumerate(selected_x0):
        closest_row = iparams.iloc[(iparams['x0'] - x0).abs().argsort()[:1]].copy()
        closest_row['near_x0'] = str(i)
        selected_params = pd.concat([selected_params, closest_row])
    selected_params = selected_params.reset_index(drop=True)

    # also select the corresponding rows from the pool_df
    selected_pool_df = pool_df.loc[pool_df['seq_id'].isin(selected_params['seq_id'])].copy()
    selected_pool_df['group'] = selected_params['group'].values[0]
    
    # also map the near_x0 values to the pool_df
    selected_pool_df['near_x0'] = selected_pool_df['seq_id'].map(
        selected_params.set_index('seq_id')['near_x0']
    )
    display(selected_params)

    # get the trajectory estimated by the model based on sequence parameters
    x = np.arange(0, 90+1, 1)
    y = {}
    for near_x0 in selected_params.near_x0.unique():
        x0, eff = selected_params.loc[selected_params.near_x0 == near_x0, "x0"].values[0], selected_params.loc[selected_params.near_x0 == near_x0, "eff"].values[0]
        y[near_x0] = [x0 * (eff)**float(c) for c in x]
    model_df = pd.DataFrame({"n_cycles": x})
    for near_x0 in y:
        model_df[near_x0] = y[near_x0]

    # melt the dataframe
    model_df = model_df.melt(id_vars="n_cycles")

    # plottin'
    fig = px.scatter(
        selected_pool_df,
        x="n_cycles",
        y="value",
        color="near_x0",
        color_discrete_map=colormap,
    )

    fig.add_traces(
        px.line(
            model_df,
            x="n_cycles",
            y="value",
            color="variable",
            color_discrete_map=colormap_light,
        ).data[:]
    )
    fig.data = fig.data[::-1]

    fig.update_layout(
        xaxis_title="PCR cycles",
        yaxis_title="Relative coverage",
        margin=dict(l=0, r=5, t=10, b=20),
        height=175,
        width=225,
        showlegend=False,
    )

    fig.update_traces(
        marker_size=6,
    )

    fig.update_yaxes(
        range=[0, 2.5],
        dtick=1,
        minor_dtick=0.5,
    )

    fig.update_xaxes(
        range=[0, 95],
        dtick=15,
        minor_dtick=15,
    )


    fig = plotting.standardize_plot(fig)
    fig.show()
    fig.write_image(f"SI_figure_more_model_trajectories/curves_{efficiency_name}.svg")
    